# Homework 3: Imbalanced Land-Cover Classification with Logistic Regression

## Scientific Context

In this homework, you will work with the **Statlog Landsat Satellite** dataset. Each observation represents a 3 by 3 neighborhood of Landsat multispectral pixels, and the original class label describes the land-cover class of the center pixel.

We will convert the original multiclass problem into an **imbalanced binary-classification** problem:

> Can we identify the minority land-cover class, **damp grey soil**, from satellite spectral features?

This is a useful geoscience and environmental-data problem because remote-sensing models are often used to map rare or spatially limited surface conditions. In those settings, ordinary accuracy can be misleading: a model may look accurate while failing to detect the class we care about.

## Learning Objectives

By completing this assignment, you will be able to:

1. Load and inspect a public earth-observation dataset in Python.
2. Convert a multiclass land-cover problem into an imbalanced binary-classification problem.
3. Quantify class imbalance and explain why accuracy can be misleading.
4. Create stratified train, validation, and test sets.
5. Build leakage-safe preprocessing and logistic-regression pipelines with `scikit-learn`.
6. Compare a naive baseline, ordinary logistic regression, and upsampled logistic regression.
7. Evaluate imbalanced classifiers using recall, precision, specificity, balanced accuracy, F-scores, ROC-AUC, and PR-AUC.
8. Evaluate whether changing the probability threshold helps, using validation data only.

## Submission

Submit this completed Jupyter notebook with all cells run and outputs visible. Replace every **TODO** with your own code or written answer. Keep the test set untouched until the final evaluation section.

## 0. Setup

This homework uses common Python machine-learning tools:

- `pandas` and `numpy` for data handling.
- `matplotlib` for plotting.
- `scikit-learn` for splitting, preprocessing, logistic regression, metrics, and pipelines.
- `imbalanced-learn` for upsampling inside a leakage-safe modeling pipeline.

Install missing packages if needed:

```bash
pip install pandas numpy matplotlib scikit-learn imbalanced-learn
```

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
POSITIVE_CLASS = 4
TARGET = "damp_grey_soil"

## 1. Load the Dataset

The dataset has 36 numeric predictors. These represent four spectral bands measured across a 3 by 3 pixel neighborhood.

The original land-cover classes are:

| Class | Land-Cover Label |
|---:|---|
| 1 | red soil |
| 2 | cotton crop |
| 3 | grey soil |
| 4 | damp grey soil |
| 5 | soil with vegetation stubble |
| 7 | very damp grey soil |

For this homework, class `4`, damp grey soil, is the positive class.

In [ ]:
feature_names = [f"band_{i}" for i in range(1, 37)]
column_names = feature_names + ["land_cover"]

train_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.trn"
test_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.tst"

landsat_raw = pd.concat(
    [
        pd.read_csv(train_url, sep=r"\s+", header=None, names=column_names),
        pd.read_csv(test_url, sep=r"\s+", header=None, names=column_names),
    ],
    ignore_index=True,
)

landsat = landsat_raw.copy()
landsat[TARGET] = np.where(
    landsat["land_cover"] == POSITIVE_CLASS,
    "damp_grey_soil",
    "other",
)
landsat = landsat.drop(columns="land_cover")

landsat.head()

### Question 1: Data Meaning and Prediction Target

Answer in 1-3 sentences.

- What is the model trying to predict?
- Why is this a classification problem rather than a regression problem?

**Answer:**  
TODO

## 2. Initial Data Checks

Before modeling, inspect the dataset. Verify the number of rows and columns, check data types, look for missing values, and summarize the numeric predictors.

In [ ]:
# TODO: inspect the dataset shape, column names, data types, missing values, and numeric summaries.

# Hints: shape, columns, dtypes, isna, describe

### Question 2: Predictor Variables

- Are the predictors mostly numeric or categorical?

**Answer:**  
TODO

## 3. Quantify Class Imbalance

Calculate the number and percentage of observations in each class. Then calculate the majority-to-minority ratio.

In [ ]:
# Calculate class counts and percentages.

landsat[TARGET].value_counts()
landsat[TARGET].value_counts(normalize=True) * 100

In [ ]:
# Calculate the majority-to-minority class ratio.

class_counts = landsat[TARGET].value_counts()
class_counts.max() / class_counts.min()

### Question 3: Why Accuracy Can Mislead

Answer in 2-3 sentences.

- If a model predicted the majority class for every row, why might its accuracy look acceptable?
- What would that model's recall be for damp grey soil?
- Why is recall important when mapping a rare class?

**Answer:**  
TODO

## 4. Create Train, Validation, and Test Sets

Use three disjoint subsets:

- **Training set:** fit preprocessing parameters and model parameters.
- **Validation set:** compare models and select a probability threshold.
- **Test set:** estimate final performance once, after all modeling choices are locked.

Use a 60% / 20% / 20% split. Because the positive class is uncommon, use stratification.

In [ ]:
X = landsat.drop(columns=TARGET)
y = landsat[TARGET]

# TODO: create X_train, X_val, X_test, y_train, y_val, and y_test.
# Use train_test_split() twice and stratify by the target.

# Hints:
# dev/test split: test_size=0.20
# train/validation split from development data: test_size=0.25
# use random_state=RANDOM_STATE

In [ ]:
# A split summary showing rows, positive count, and positive fraction
# for train, validation, and test.

split_summary = pd.DataFrame(
    {
        "rows": [len(y_train), len(y_val), len(y_test)],
        "positive_count": [
            (y_train == "damp_grey_soil").sum(),
            (y_val == "damp_grey_soil").sum(),
            (y_test == "damp_grey_soil").sum(),
        ],
        "positive_fraction": [
            (y_train == "damp_grey_soil").mean(),
            (y_val == "damp_grey_soil").mean(),
            (y_test == "damp_grey_soil").mean(),
        ],
    },
    index=["train", "validation", "test"],
)

split_summary

### Question 4: Splitting and Leakage

Answer each prompt in 2-4 sentences.

1. Why is stratification useful in this dataset?
2. What is the validation set used for?
3. Why should the test set not be used to choose a model or threshold?
4. Explain how fitting scaling, imputation, or upsampling before splitting can leak information.

**Answers:**

1. TODO
2. TODO
3. TODO
4. TODO

## 5. Build Logistic-Regression Pipelines

We will compare three pipelines:

1. A majority-class baseline model.
2. Ordinary logistic regression.
3. Logistic regression with upsampling applied inside the pipeline.

Upsampling increases the number of minority-class examples in the training data by resampling them. It must occur inside the modeling pipeline so that it is applied only to training data.

In [ ]:
# Create a basic preprocessing pipeline for logistic regression.

preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

In [ ]:
# Create the baseline and logistic-regression model pipelines.

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)

upsampled_logistic_pipeline = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("upsampler", RandomOverSampler(random_state=RANDOM_STATE)),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)

In [ ]:
# TODO: fit all three pipelines on the training data.

# Hints:
# baseline_fit = baseline_pipeline.fit(...)
# logistic_fit = logistic_pipeline.fit(...)
# upsampled_logistic_fit = upsampled_logistic_pipeline.fit(...)

### Question 5: Model Choice

Answer in 1-3 sentences.

- Why include a majority-class baseline?
- Why is logistic regression a reasonable first classification model?
- What does upsampling change?
- Why must upsampling happen inside the training pipeline?

**Answer:**  
TODO

## 6. Evaluate Models on the Validation Set

For an imbalanced classification problem, no single metric is sufficient.

Important metrics:

- **Recall / sensitivity:** fraction of damp-grey-soil pixels detected.
- **Precision:** fraction of predicted damp-grey-soil pixels that were actually damp grey soil.
- **Specificity:** fraction of other land-cover pixels correctly identified.
- **Balanced accuracy:** mean of sensitivity and specificity.
- **F1:** balances precision and recall equally.
- **F2:** gives recall more weight than precision.
- **ROC-AUC:** ranking performance across thresholds.
- **PR-AUC:** precision-recall area; often useful for imbalanced data.

In [ ]:
positive_level = "damp_grey_soil"


def positive_probability(model, X_data):
    """Return predicted probability for the positive class."""
    probabilities = model.predict_proba(X_data)
    positive_index = np.where(model.classes_ == positive_level)[0][0]
    return probabilities[:, positive_index]


def predict_with_threshold(model, X_data, y_true, threshold=0.5):
    # TODO: return a DataFrame with truth, predicted probabilities, and thresholded classes.

    # Hints:
    # probabilities = positive_probability(...)
    # predictions = np.where(probabilities >= threshold, "damp_grey_soil", "other")
    # return pd.DataFrame({...})
    pass


def calculate_metrics(predictions):
    y_true = predictions["truth"]
    y_pred = predictions["prediction"]
    y_prob = predictions["probability"]

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=["other", "damp_grey_soil"],
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "accuracy": (y_true == y_pred).mean(),
        "recall": recall_score(y_true, y_pred, pos_label=positive_level, zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=positive_level, zero_division=0),
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, pos_label=positive_level, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, pos_label=positive_level, zero_division=0),
        "roc_auc": roc_auc_score(y_true == positive_level, y_prob),
        "pr_auc": average_precision_score(y_true == positive_level, y_prob),
    }

In [ ]:
# TODO: calculate validation metrics for all three models at threshold 0.5.

# Hints:
# baseline_val = predict_with_threshold(...)
# logistic_val = predict_with_threshold(...)
# upsampled_val = predict_with_threshold(...)
# pd.DataFrame([...], index=[...])

In [ ]:
# TODO: create confusion matrices for the two logistic-regression models.

# Hint:
# ConfusionMatrixDisplay.from_predictions(...)

### Question 6: Validation Comparison

Using the validation results, answer in complete sentences.

1. Why can the baseline model have high accuracy but zero usefulness for the positive class?
2. Compare ordinary logistic regression and upsampled logistic regression at threshold 0.5.
3. Which model has better recall for damp grey soil?
4. What happened to precision and specificity?

**Answers:**

1. TODO
2. TODO
3. TODO
4. TODO

## 7. Examine ROC and Precision-Recall Curves

Curves show model behavior across possible thresholds. For imbalanced classification, the precision-recall curve is often more informative than the ROC curve because it focuses directly on positive-class detection.

In [ ]:
# Plot ROC curves for ordinary and upsampled logistic regression.

for name, model in {
    "Logistic regression": logistic_fit,
    "Upsampled logistic regression": upsampled_logistic_fit,
}.items():
    probabilities = positive_probability(model, X_val)
    fpr, tpr, _ = roc_curve(y_val == positive_level, probabilities)
    auc = roc_auc_score(y_val == positive_level, probabilities)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="No-skill")
plt.xlabel("False-positive rate")
plt.ylabel("True-positive rate / recall")
plt.title("Validation ROC curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# TODO: plot precision-recall curves for ordinary and upsampled logistic regression.

# Hints:
# precision, recall, _ = precision_recall_curve(...)
# average_precision_score(...)
# plt.plot(recall, precision, ...)
# plt.axhline((y_val == positive_level).mean(), linestyle="--")

### Question 7: Curve Interpretation

Answer in 3-6 sentences.

- Why is the precision-recall baseline equal to the positive-class prevalence?
- Can two models have similar ROC-AUC but different rare-class precision? Why?
- Which curve is more useful for thinking about missed rare land-cover areas versus false alarms?

**Answer:**  
TODO

## 8. Test Probability Thresholds Using Validation Data

The default threshold of 0.5 is not automatically best for an imbalanced problem, but changing the threshold does not improve the model itself. It only changes the tradeoff between the types of errors the model makes.

For this exercise, test a range of thresholds for the **upsampled logistic regression** model and identify the threshold that maximizes **F2** on the validation set. Then compare that threshold with 0.5. If the F2-selected threshold creates too many false positives, lowers balanced accuracy, or makes the model less useful for the stated mapping goal, it is reasonable to conclude that threshold adjustment did not help.

The important lesson is not that threshold tuning always improves performance. The lesson is that thresholds encode a decision policy: they decide whether the model should miss more positive cases or produce more false alarms.

In [ ]:
# Search thresholds from 0.01 to 0.99 and identify the threshold with the best F2.

threshold_grid = np.arange(0.01, 1.00, 0.01)

threshold_rows = []
for threshold in threshold_grid:
    predictions = predict_with_threshold(
        upsampled_logistic_fit,
        X_val,
        y_val,
        threshold=threshold,
    )
    row = {"threshold": threshold, **calculate_metrics(predictions)}
    threshold_rows.append(row)

threshold_results = pd.DataFrame(threshold_rows)
best_index = threshold_results["f2"].idxmax()
best_threshold = threshold_results.loc[best_index, "threshold"]

threshold_results.loc[[best_index]]

In [ ]:
# TODO: plot precision, recall, and F2 across thresholds.

# Hint:
# plt.plot(threshold_results["threshold"], threshold_results["precision"], label="Precision")
# plt.plot(...)
# plt.axvline(best_threshold, linestyle="--")

### Question 8: Threshold Choice

Answer in complete sentences.

1. Is your selected threshold above or below 0.5?
2. What happens to false positives and false negatives when the threshold is lowered?
3. Did the F2-selected threshold make the model more useful than the 0.5 threshold? Explain using at least two metrics or confusion-matrix counts.
4. Is maximizing F2 automatically the correct policy? What field, cost, or scientific information would you need before choosing a real operational threshold?

**Answers:**

1. TODO
2. TODO
3. TODO
4. TODO

## 9. Final Evaluation on the Untouched Test Set

At this point, the model and threshold have been selected using validation data. If the F2-selected threshold made validation performance worse for the assignment goal, you may keep the default 0.5 threshold instead. State which threshold you chose and why.

Do not use the test result to go back and choose a different model or threshold.

In [ ]:
# Evaluate the upsampled logistic regression model on the test set
# using the threshold chosen from validation data.

final_test_predictions = predict_with_threshold(
    upsampled_logistic_fit,
    X_test,
    y_test,
    threshold=best_threshold,
)

calculate_metrics(final_test_predictions)

ConfusionMatrixDisplay.from_predictions(
    final_test_predictions["truth"],
    final_test_predictions["prediction"],
    labels=["other", "damp_grey_soil"],
    display_labels=["Other", "Damp grey soil"],
    values_format="d",
)
plt.title("Final test confusion matrix")
plt.tight_layout()
plt.show()

### Question 9: Methods and Limitations

Answer in a short paragraph.

1. Compare upsampling with class weighting and random undersampling.
2. Where must resampling occur to avoid leakage when cross-validation is used?

**Answer:**  
TODO

## Grading Rubric (25 points)

| Component | Points |
|---|---:|
| Reproducibility, loading, and data checks | 2 |
| Class-imbalance analysis | 3 |
| Correct train/validation/test split and leakage explanation | 3 |
| Logistic-regression pipelines | 3 |
| Baseline and model fitting | 2 |
| Validation metrics and interpretation | 3 |
| ROC and precision-recall curve interpretation | 2 |
| Threshold comparison using validation data | 2 |
| Final test evaluation | 2 |
| Scientific interpretation and limitations | 3 |

## Reproducibility Checklist

Before submitting, confirm that:

- [ ] The notebook runs without manual state from earlier sessions.
- [ ] All `TODO` sections are complete.
- [ ] Random seeds are fixed.
- [ ] Preprocessing is fitted only through pipelines on training data.
- [ ] Upsampling occurs only inside the pipeline.
- [ ] Validation data, not test data, are used for model and threshold decisions.
- [ ] The test set is evaluated only in the final section.
- [ ] Metrics are interpreted for the positive class, damp grey soil.
- [ ] Figures have readable titles and axis labels.
- [ ] Scientific limitations and error costs are discussed.

## Dataset Citation

Srinivasan, A. (1993). Statlog (Landsat Satellite) [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C55887